# Day 1: getting to know the data

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

## The question

Was summer 2026 actually unusual, or does it just feel that way? Over two sessions we answer it with the Met Office's monthly weather record for Oxford, which starts in 1853. Today we get the data into shape and take a first look. Tomorrow we plot it, fit a trend and bring in a second station.

## By the end of today you will be able to

- read a messy text file into a pandas DataFrame, choosing the separator, header and missing-value markers yourself
- explain how one stray character turns a column of numbers into text, and fix it without throwing information away
- summarise every column of a table at once with `.describe()`
- turn a column into dates, and pick rows by their label with `.loc` and by their position with `.iloc`
- pick out the rows and columns that answer a question, combining conditions with `&`, `|` and `~`
- say what a NumPy array is, why vectorised code beats a loop, and how NaN behaves in NumPy and pandas
- decide what to do about missing data, and defend the decision
- summarise data with split-apply-combine (`groupby`)

## How to use this notebook

There are two versions of this notebook. In `day1.ipynb` the explanations are written for you and most code cells are empty: in the session, you type the code into them as the instructor types it on screen. A few cells, mostly setup, are filled in already; just run those. [`day1_solutions.ipynb`](day1_solutions.ipynb), in the same folder, has every cell filled in, with its output. If you fall behind, copy the cell you missed from there and carry on. Afterwards, either one is your notes.

The [CfRR Python for Data Analysis pages](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/section_landing_pages/python_for_data_analysis.html) are the full reference. This notebook links to them when you might want every option.

*Data: Met Office historic station data for Oxford. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

## 1. Welcome, the question and the data

*10:00, 10 minutes*

Before any code: was summer 2026 in Oxford unusual?

### Meet the data

The Met Office publishes [monthly records](https://www.metoffice.gov.uk/research/climate/maps-and-data/historic-station-data) for a set of long-running UK weather stations: one plain text file per station, free to reuse. We will use Oxford, whose record starts in January 1853. The station is at 51.76°N, 1.26°W, 63 metres above sea level.

Each month in the file is summarised by five numbers:

| Column | What it measures | Units |
|---|---|---|
| `tmax` | Mean daily maximum temperature: each day's highest temperature, averaged over the month | °C |
| `tmin` | Mean daily minimum temperature: each night's lowest temperature, averaged over the month | °C |
| `af` | Days of air frost: days when the air temperature fell below 0 °C | days |
| `rain` | Total rainfall for the month | mm |
| `sun` | Total hours of sunshine in the month | hours |

Two things are easy to misread:

- **`tmax` is an average, not a record.** A July `tmax` of 22 °C means the daily highs averaged 22 °C. Some days were much hotter.
- **"Summer" means June, July and August.** That is the meteorologists' definition, and the one we will use.

The record is not perfect (data very rarely is!):

- **Estimated** values fill gaps using the Met Office's gridded dataset (HadUK-Grid) rather than a reading at the station.
- **Missing** values are months with more than two days of data missing.
- **Provisional** values have not yet been through the Met Office's full quality control, and may be revised.
- **Sunshine** was measured for most of the record by a Campbell-Stokes recorder, a glass ball that focuses the sun to burn a trace on a card. Some stations now use an automatic sensor instead.
- Over 170 years, the site and instruments may have changed slightly.

Knowing what each number means, and how it was made, is part of the analysis: it decides which questions the data can answer.

## 2. Python warm-up

*10:10, 10 minutes*

A quick reminder of the Python from the Introduction to Python course, before we need it. Every idea here comes back later today or tomorrow, on real data.

A notebook runs cells in the order you run them, not the order they appear on the page. Run a cell with Shift+Enter. If things get confusing, use **Kernel > Restart Kernel and Run All Cells**.

Every value has a **type**. Text is a string, `str`, and a string has **methods**: functions attached to it, called with a dot. Here is a temperature as it appears in today's file, with a marker on the end:

`.endswith("*")` answers `True` or `False`, a `bool`. `.rstrip("*")` strips the `*` from the end, but the result is still text. `float` turns it into a number you can do arithmetic with. Comparisons give more `bool`s, and `and`, `or` and `not` combine them:

A **list** holds values in order. Square brackets pick one out by its position, counting from 0, and a `for` loop visits each in turn:

A **dictionary**, `dict`, looks values up by a key instead of a position:

A **function** packages steps you want to reuse. `def` gives it a name and its inputs, the indented lines are its body, and `return` hands back the result:

You will meet all of these again today: a list of column names in block 4, string methods and a loop in block 5, and a dictionary and `and` in block 6. Tomorrow you will write a function that loads any station's file.

## 3. First look at the raw file

*10:20, 10 minutes*

### The tools

- **pandas** is the standard Python library for tables of data. Its central object is the **DataFrame**: rows and named columns, like a spreadsheet that you drive with code. Every step is written down, so anyone can rerun it and get the same result, which is hard to promise for clicks in a spreadsheet.
- **NumPy** does the fast arithmetic underneath pandas. We look at it properly later this morning.
- **pathlib** builds file locations that work on Windows, macOS and Linux alike.

`import pandas as pd` and `import numpy as np` are conventions: almost all the pandas code you will find elsewhere uses the same short names.

The data came in the zip with this notebook, in the `data` folder next to it. It is a dated snapshot of the Met Office file rather than the live one, so everyone works with exactly the same numbers, and they stay the same when the Met Office updates its own file. This first cell is filled in for you: run it. It should show `True`, meaning Python found the file.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/oxforddata.txt
OXFORD = DATA / "oxforddata.txt"
OXFORD.exists()

Look at a data file before you load it. In the file browser on the left, open the `data` folder and double-click `oxforddata.txt`. JupyterLab opens it as plain text in a new tab. Use JupyterLab rather than Excel, which would try to split it into columns for you.

Read the top, then jump to the bottom with Ctrl+End (Cmd+Down on a Mac). What will make this file awkward to load into a table?

Six things about this file will make it awkward to load. How many did you find?

- five lines of free text before the column names
- a row of units under the column names
- columns separated by spaces rather than commas
- `---` for missing values
- a `*` after estimated values
- the word `Provisional` at the end of recent rows

Each one causes a problem in the next half hour. The obvious first try is `pd.read_csv` with no options at all:

One column. `read_csv` splits on commas by default and there are none, so each whole line becomes a single value, and the first line (`Oxford`) becomes the column name.

## 4. Loading it properly

*10:30, 15 minutes*

"CSV" means comma-separated values, but `read_csv` reads any delimited text once you tell it how. Reading a file is a set of decisions, not a default:

- **`sep`**: what separates the columns
- **`skiprows`**: how many lines to skip before the data
- **`names`**: what to call the columns
- **`na_values`**: how this file writes "missing"

The [CfRR Pandas page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Pandas.html) covers the other options.

First the separator. `sep=r"\s+"` means "one or more whitespace characters".

An error. Read error messages from the bottom up: the last line says what went wrong, and everything above it says where. Here pandas expected 10 fields on line 4 and found 13. Line 4 is free text, not data.

So we skip all seven lines before the first row of numbers, and supply the column names ourselves. Remember the word `Provisional` at the end of recent rows: those rows have an eighth field, so the data are ragged, with some rows longer than others. That word tells us these numbers may still change, so rather than throw it away we give it a column of its own, `status`.

It loads, and the last rows show `Provisional` in `status`. Now look at `sun` in the first rows: `---` is this file's way of writing "missing", and pandas has kept it as text. `na_values` tells pandas to read it as missing instead:

### Anatomy of a DataFrame

Now the data are in, it is worth knowing the parts:

- the **columns**, each with a name
- the **index**: the row labels down the left (for now just 0, 1, 2 and so on)
- each column on its own is a **Series**: one column of values that keeps the index

`shape` gives the number of rows and columns, and `columns` their names. Selecting one column gives a Series: look at the last line of its output, `dtype`. We will need it soon.

`na_values="---"` turned the file's own missing marker into proper missing values (`NaN`). Count them in each column:

`sun` has the most: sunshine was not recorded at Oxford until 1929. `status` is mostly missing too, but that is expected, because only provisional rows have a status. One month, December 1860, is missing `tmin` and `af`.

## 5. The asterisk problem

*10:45, 20 minutes*

The first thing anyone wants from a column of temperatures is its average.

It fails. `.info()` summarises the whole DataFrame: every column, how many values it has (`Non-Null Count`) and its data type (`Dtype`):

`tmax` is stored as text (`str` in pandas 3, `object` in earlier versions). One value pandas cannot read as a number is enough to make the whole column text. Here are the culprits:

The `*` marks an estimated value. That matters: July 2026, the month we care about, is one of them.

### The quick fix, and why it is wrong

`pd.to_numeric` converts text to numbers. On its own, it stops at the first value it cannot read:

Ask an AI assistant, or search online, and you will often be told to add `errors="coerce"`. Try it:

No error, and an average that looks sensible. Now count the missing values it leaves:

`errors="coerce"` turns anything it cannot read into `NaN`, without telling you. Every estimated value disappears, including July 2026. The code runs and the answer is wrong. That is the worst kind of bug.

A better approach keeps the information: record which values were estimated in a column of its own, then strip the `*` and convert. `.astype(float)` fails loudly if it meets anything else it does not expect, which is what we want. `.str.rstrip("*")` strips a `*` from the end of each value, where this file puts it; `.str.replace("*", "")` would remove one from anywhere, and works here too.

Now the mean includes every month. Compare it with the 13.98 °C from `errors="coerce"`:

Higher. Most estimated values are recent, warm months, so dropping them quietly pulled the average down.

All five measured columns need the same treatment, and the results should go back into `df`. A loop does it once for all of them: it replaces each column with its cleaned version and adds an `_estimated` column alongside it.

`astype(str)` first makes sure every column is text before we look for a `*`. At some stations a column has no estimates at all, so pandas reads it as numbers already, and `.str` only works on text. The last line, `df.dtypes`, lists just the data types, so we can check that the five measurements are now numbers (`float64`).

The `status` column gets the same idea: a `True`/`False` column called `provisional` keeps the information, and then `status` can go.

We have kept every flag as its own column rather than deleting the estimated and provisional rows. Throwing them away would have thrown away July 2026.

### Summarising every column

Now that every measurement is a number, `.describe()` summarises them all in one table: how many values each column has (`count`), the mean and the standard deviation (`std`), the smallest and largest values, and the quartiles in between. `.round(1)` keeps it readable.

A lot can be read off this one table:

- **count** shows the gaps: `sun` has far fewer values than the other columns, and `tmin` and `af` are one short.
- **min** and **max** are a quick check that nothing is impossible. The hottest month, 28.2 °C, is July 2026.
- **50%** is the median: in half of all months, less than 49.9 mm of rain fell.

`.describe()` is worth running on any dataset as soon as it loads properly.

## Break

*11:05, 10 minutes*

## 6. Picking rows: dates, labels and masks

*11:15, 25 minutes*

Most questions about data start with "which rows?": which month was the hottest, which winters had no frost. pandas has three ways to pick rows: by their **label** in the index, by their **position**, and with a **mask** that tests every row. Labels are only useful if they mean something, so first give every row a proper date.

### A date for every row

`pd.to_datetime` makes dates. Given a dictionary of years, months and days, it builds one date for each row. Our rows are whole months, so use the 1st as the day:

The last line of the output says `datetime64`: pandas' type for dates and times. `yyyy` and `mm` are just numbers, but a `datetime64` column knows it holds dates. The `.dt` accessor reaches the parts of each date, as `.str` reached into text:

Dates in most files arrive as text, such as `01/07/2026`. `pd.to_datetime` reads text too, but is that 1 July or 7 January? Unless told, pandas assumes the American order, month first. `format` says exactly how to read it: `%d` is the day, `%m` the month and `%Y` the four-digit year.

Without `format`, 7 January, and no error. Whenever dates arrive as text, give `format`: pandas then reads them the way you mean, and fails loudly on any value that doesn't match. The [CfRR Pandas page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Pandas.html) uses it on dates with times.

### Dates as the index

`set_index` moves a column into the index, so the dates become the row labels down the left. `reset_index` does the reverse.

`.loc` picks rows by their label. With dates as labels, you can write a month (`"2026-07"`), a year (`"1976"`) or a range of dates as text. Summer 1976, a famous heatwave:

Unlike a Python slice, a `.loc` range includes its last label, so August is in. `.iloc` ignores the labels and counts positions from 0, like a list, so `df.iloc[0]` is the first row whatever its label:

January 1853. Before `set_index` the labels were 0, 1, 2 and so on, so `df.loc[0]` and `df.iloc[0]` were the same row. Now `df.loc[0]` fails: no row is labelled 0 any more.

### Masks: one condition

A **boolean mask** picks rows by testing every one of them: a column of `True`/`False` values, one for each row, saying whether that row passes a test. You have made masks already: `df["tmax"].str.endswith("*")` in block 5 was one, and each `_estimated` column is one kept in the DataFrame. After the second break you will practise with masks, and everything else from today, on your own.

The simplest mask is one comparison on a column. When you add up a mask, `True` counts as 1, so `.sum()` counts the rows that pass:

For the same reason, `.mean()` of a mask is the fraction of rows that pass:

Just under half of all months had no air frost.

### Rows and columns together

`df.loc[rows, columns]` picks rows and columns at once. The rows can be a mask. The columns can be one name, or a list of names. A list has its own square brackets, which is why you have already seen double brackets, as in `df[["yyyy", "mm"]]`.

### Combining conditions

Python's own `and`, `or` and `not` don't work on whole columns of `True`/`False`. pandas uses three symbols instead:

- `&` (and): true where both masks are true
- `|` (or): true where at least one is true
- `~` (not): flips `True` and `False`

Which hot months are estimated? Put each comparison in its own brackets:

Five months, all from the last two summers: the hottest months in the record are also the least certain.

Now the same question with the brackets left out:

2083 of the 2085 months, and no error. Python works out `&` before `>`, so this compares `tmax` with `23 & df["tmax_estimated"]`, which is 1 or 0, and almost every month is warmer than that. Like `errors="coerce"`, it runs and the answer is wrong. Put every comparison in brackets, and check that the number of rows makes sense.

`|` (or) works just like `&`, with brackets round each comparison; you will use it in the practice. `~` flips a mask, so `~df["tmax_estimated"]` is every month that was measured rather than estimated. The hottest of those:

27.4 °C, July 2018: the record if you set the estimates aside.

### A shortcut: `.isin()`

`.isin()` is true where a value is one of a list, so `df["mm"].isin([12, 1, 2])` does the job of three `==` comparisons joined with `|`. It is a method rather than a comparison, so it needs no extra brackets. Winter months with an average maximum above 11 °C:

All seven are from 1990 onwards.

## 7. What's underneath: NumPy

*11:40, 20 minutes*

Every pandas column is a NumPy array with a name and an index attached. NumPy is what makes the arithmetic fast. The [CfRR NumPy page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_NumPy.html) has the full tour.

The mean temperature for a month is the average of its mean maximum and mean minimum. Here it is the way the Introduction to Python course would write it, with a loop:

NumPy can do the same calculation without a loop. Arithmetic on an array applies to every element at once: this is called **vectorisation**.

Same numbers, less code. It is also much faster. `%timeit` runs a line many times and reports how long it takes:

The vectorised version is many times faster, and the gap grows with the size of the data. That is why pandas and NumPy code avoids looping over rows.

### How NaN behaves

December 1860 has no `tmin`. Here are three ways to average `tmin`, missing month included:

`np.mean` returns `nan`: any arithmetic with `NaN` gives `NaN`, so one missing month spoils the whole result. `np.nanmean` skips missing values. pandas skips them by default. Neither choice is wrong, but you need to know which one you are getting.

Add the monthly mean temperature to the DataFrame. pandas lines the values up by row for us.

### Missing data is a decision

`df.dropna()` removes every row with a missing value; `df.dropna(subset=["tmin"])` would only look at the columns you list. How many rows would survive?

Only just over half. Sunshine is missing for every month from 1853 to 1928, so dropping those rows also throws away 76 years of perfectly good temperature records.

There are four things we could do about the missing sunshine, and each has a cost:

1. **Drop every row with a missing value.** Loses most of the temperature record from before 1929.
2. **Drop the `sun` column.** Loses the sunshine we do have.
3. **Fill the gaps with the average sunshine.** Invents data, as the next cell shows.
4. **Keep everything, leave the gaps as `NaN`, and only use sunshine from 1929.**

We take option 4. pandas already skips `NaN` in calculations, so the gaps do no harm as long as we know they are there. Option 3 is the most dangerous, because it looks harmless:

Filling invents 76 years of perfectly average sunshine, and makes the record look far less variable than it was. Any analysis of sunshine over time would be quietly wrong.

The single missing `tmin` can stay as `NaN` too. Filling one month in a long record would also be defensible, as long as you say so.

## 8. Summarising: split-apply-combine

*12:00, 15 minutes*

Before writing code, say the plan in plain English. To get the average summer (June to August) maximum temperature for each year:

1. keep only the June, July and August rows
2. **split** them into groups by year
3. **apply** the mean to `tmax` in each group
4. **combine** the results into one table

That is split-apply-combine, and `groupby` does the last three steps in one line.

Group by month instead of year, using all the rows, and the same line answers a different question:

This is the average maximum temperature for each month of the year over the whole record: the seasonal cycle.

Now rank every July in the record by `tmax`, hottest first, and show the top five with their flags:

## 9. Cliffhanger

*12:15, 5 minutes*

Where does summer 2026 come in the ranking of whole summers?

So July 2026 is the hottest July on record, and summer 2026 the hottest summer. Is that enough to say it was unusual?

- The 2026 temperatures are estimated and provisional.
- In any long record, some year has to be the hottest. Is 2026 far ahead, or just a bit ahead?
- Summers have been getting warmer. Is 2026 unusual for its time, or just the latest step on a trend?

Tomorrow we plot it, fit a trend and find out.

### Today's ideas

- Reading a file is a set of decisions: `sep`, `skiprows`, `names`, `na_values`.
- One stray character turns a numeric column into text. Fix it and keep the information as a flag.
- `errors="coerce"` hides problems. Prefer code that fails loudly.
- `.describe()` summarises every column in one table. Run it on any new dataset.
- `pd.to_datetime` makes real dates; give it a `format` when dates are text. With dates as the index, `.loc` picks rows by label and `.iloc` by position.
- A mask is a column of `True`/`False` that picks rows. Combine masks with `&`, `|` and `~`, with every comparison in brackets.
- A column is a NumPy array. Vectorised arithmetic beats loops.
- `NaN` spreads in NumPy and is skipped in pandas.
- Dropping or filling missing data is a decision with consequences.
- `groupby` is split-apply-combine.

## Break

*12:20, 10 minutes*

## 10. Practice

*12:30, 30 minutes*

Time to use everything from today on your own. The questions are grouped under the parts of the day they practise. Try at least one under each heading, then as many more as you like, in any order. There are more than you will finish today: carry on with the rest in your own time.

Write each answer in the empty cell under its question. The answers are in `day1_solutions.ipynb`, but try each one yourself first. If a mask is not doing what you expect, run it on its own and check its `.sum()`.

### Loading the file

**1.** `read_csv` has an option, `nrows`, that reads only that many rows of data. Read just the first two years (24 months) of the Oxford file into `first_two`, with the same separator, skipped lines, column names and missing-value marker as this morning. What is the dtype of its `tmax` column, and why isn't it text this time?

### Cleaning and summarising

**2.** How many values in each measurement column are estimated? Pick out the five `_estimated` columns with a list of names, then use `.sum()`.

**3.** `.rename(columns={...})` gives columns new names, with a dictionary from each old name to its new one. Make `readable`, a copy of `df` with `af` renamed to `frost_days` and `sun` to `sunshine_hours`, and show the last three rows of those two columns. Does `df` still have a column called `af`?

**4.** Use `.describe()` to compare `rain` in summer (June to August) with winter (December to February). Which season is wetter on average, and which varies more?

### Dates and the index

**5.** Use `.loc` and a date to show `tmax`, `rain` and `sun` for January 2000, or for the month you were born. Then use `.iloc` to find the 1000th month in the record. Which month is it?

**6.** The winter of 1962 to 1963, the "Big Freeze", was one of the coldest in English history. Use a `.loc` range of dates to pick out December 1962 to February 1963. What was its mean `tmax`, and how many days of air frost did it have in total?

### Masks

**7.** Make a DataFrame of the months when `tmin` was below 0 °C, with only the `yyyy` and `sun` columns. How many months are there, and why is `sun` missing for so many of them?

**8.** What was the median rainfall in months when `tmax` was above 20 °C? How does it compare with the median for all months?

**9.** Hot and dry: how many months had a `tmax` above 22 °C and less than 20 mm of rain? How many of them were in 2000 or later?

**10.** How many months were either very wet (more than 150 mm of rain) or very dry (less than 5 mm)? Combine the two conditions with `|`.

**11.** Climate scientists compare each month with its average over 30 years, often 1991 to 2020. `.between(low, high)` is true where a value is in a range, both ends included. Use it to find the mean July `tmax` from 1991 to 2020. How far above it was July 2026, at 28.2 °C?

### NumPy and missing values

**12.** Convert `tmax` to degrees Fahrenheit for every month at once, without a loop. The warm-up's `to_fahrenheit` works on a whole column too: try it. What is the hottest month in the record in °F?

**13.** In which year does the sunshine record start? Use `~` and `.isna()` to keep only the months with sunshine, then find the smallest `yyyy`.

### Split-apply-combine

**14.** Which month of the year is the wettest on average, and which is the driest?

**15.** Which five years had the most days of air frost in total?

### Your own question

**16.** Write a question of your own about the data that needs at least two conditions, and answer it. The solutions notebook has one example.